In [3]:
import json
import pandas as pd

INPUT_JSON = "gen8ou.json"

# Output files
USAGE_CSV = "gen8ou_usage.csv"
ITEMS_CSV = "gen8ou_items.csv"
MOVES_CSV = "gen8ou_moves.csv"
TEAMMATES_CSV = "gen8ou_teammates.csv"
ALL_WIDE_CSV = "gen8ou_all.csv"

def clean_dict(d):
    """Remove junk keys and coerce values to float."""
    if not isinstance(d, dict):
        return {}
    out = {}
    for k, v in d.items():
        if k in ("", "empty", None):
            continue
        try:
            out[str(k)] = float(v)
        except Exception:
            continue
    return out

with open(INPUT_JSON, "r", encoding="utf-8") as f:
    obj = json.load(f)

data = obj["data"]
total_battles = obj.get("info", {}).get("number of battles", 0) or 0

# pokemon -> dict of columns
usage_rows = {}
items_rows = {}
moves_rows = {}
teammates_rows = {}
all_rows = {}

for mon, info in data.items():
    if not isinstance(info, dict):
        continue

    # --- usage fields ---
    usage = float(info.get("usage", 0) or 0)
    raw_usage = float(info.get("Raw count", 0) or 0)
    real_usage = (raw_usage / total_battles) if total_battles else 0.0

    usage_rows[mon] = {
        "usage": usage,
        "raw_usage": raw_usage,
        "real_usage": float(real_usage),
        "total_battles": float(total_battles)
    }

    # --- items/moves/teammates (NO prefixes for these individual datasets) ---
    items = clean_dict(info.get("Items", {}))
    moves = clean_dict(info.get("Moves", {}))
    teammates = clean_dict(info.get("Teammates", {}))

    items_rows[mon] = items                 # columns like "heavydutyboots"
    moves_rows[mon] = moves                 # columns like "sunnyday", "pound"
    teammates_rows[mon] = teammates         # columns like "Landorus-Therian"

    # --- combined dataset (WITH prefixes only here) ---
    combined = {}
    combined.update(usage_rows[mon])
    combined.update({f"item:{k}": v for k, v in items.items()})
    combined.update({f"move:{k}": v for k, v in moves.items()})
    combined.update({f"teammate:{k}": v for k, v in teammates.items()})
    all_rows[mon] = combined

# Build DataFrames (rows=pokemon, cols=features)
usage_df = pd.DataFrame.from_dict(usage_rows, orient="index").fillna(0)
items_df = pd.DataFrame.from_dict(items_rows, orient="index").fillna(0)
moves_df = pd.DataFrame.from_dict(moves_rows, orient="index").fillna(0)
teammates_df = pd.DataFrame.from_dict(teammates_rows, orient="index").fillna(0)
all_df = pd.DataFrame.from_dict(all_rows, orient="index").fillna(0)

# Ensure numeric + missing -> 0
for df in (usage_df, items_df, moves_df, teammates_df, all_df):
    df[:] = df.apply(pd.to_numeric, errors="coerce").fillna(0)

# Add pokemon label once as first column
for df in (usage_df, items_df, moves_df, teammates_df, all_df):
    df.insert(0, "pokemon", df.index)

# Save
usage_df.to_csv(USAGE_CSV, index=False)
items_df.to_csv(ITEMS_CSV, index=False)
moves_df.to_csv(MOVES_CSV, index=False)
teammates_df.to_csv(TEAMMATES_CSV, index=False)
all_df.to_csv(ALL_WIDE_CSV, index=False)

print("✅ Wrote 5 files (rows=pokemon, cols=features; missing=0):")
print(" -", USAGE_CSV)
print(" -", ITEMS_CSV)
print(" -", MOVES_CSV)
print(" -", TEAMMATES_CSV)
print(" -", ALL_WIDE_CSV)


✅ Wrote 5 files (rows=pokemon, cols=features; missing=0):
 - gen8ou_usage.csv
 - gen8ou_items.csv
 - gen8ou_moves.csv
 - gen8ou_teammates.csv
 - gen8ou_all.csv
